# RAR Per-Seed Campaign Runner

**What this does:** Runs the RAR experiment one seed at a time, writing each seed's
result to its own file (`pilot_seed_<seed>.json`). If Colab disconnects mid-run,
completed seeds are safe on disk and in Google Drive.

**Critical safety guard:** If the API key is missing (e.g. lost on runtime restart),
the code now **raises immediately** instead of silently fabricating fake results.

**After all seeds finish**, run the merge cell at the bottom to stitch them into
one `pilot_results.json` with the real Wilcoxon p-value.

## Cell 1: Mount Google Drive (optional but recommended for persistence)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2: Clone repo & install dependencies

In [ ]:
import os

REPO_DIR = '/content/recursive-autonomy-research'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Tahir-yamin/recursive-autonomy-research.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!ls -la *.py

In [ ]:
!pip install -q torch torchvision numpy scipy scikit-learn networkx aiohttp matplotlib

## Cell 3: Set API Key & Model

**YOU MUST RUN THIS CELL BEFORE EVERY SEED.** If Colab restarts the runtime,
env vars are wiped. The guard will halt if the key is missing.

Replace `YOUR_OPENROUTER_KEY_HERE` with your actual key.

In [ ]:
import os

# ===== SET YOUR KEY HERE =====
os.environ['OPENROUTER_API_KEY'] = 'YOUR_OPENROUTER_KEY_HERE'  # <-- PASTE YOUR KEY
os.environ['OPENROUTER_MODEL']   = 'google/gemma-3-27b-it:free'

# Safety check — if you forgot to paste, this tells you now, not 40 minutes in
key = os.environ.get('OPENROUTER_API_KEY', '')
if not key or key == 'YOUR_OPENROUTER_KEY_HERE':
    raise ValueError('\n\n>>> STOP: Paste your real OpenRouter API key above! <<<\n')
else:
    print(f'API key set (ends ...{key[-6:]})')
    print(f'Model: {os.environ["OPENROUTER_MODEL"]}')

## Cell 4: Check GPU (optional)

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('No GPU — will run on CPU (slower but works fine)')

## Cell 4b: PRE-FLIGHT — live API ping

Verifies your key **and** the Gemma model actually return a response *before*
you commit to a multi-hour run. Catches a bad key, wrong model name, or empty
free-tier quota in ~2 seconds instead of 40 minutes in. Run this once per session.

In [ ]:
import os, asyncio, sys
os.chdir('/content/recursive-autonomy-research')

if 'run_pilot_experiment' in sys.modules:
    del sys.modules['run_pilot_experiment']
import run_pilot_experiment as rpe

async def _preflight():
    # One tiny real call through the same code path the campaign uses.
    return await rpe.call_llm('Reply with the single word: OK')

resp = await _preflight()
if not resp:
    raise RuntimeError(
        '\n>>> PRE-FLIGHT FAILED: API returned nothing. <<<\n'
        'Check: (1) key is valid, (2) model name is correct & available on the '
        'free tier, (3) you have remaining quota. Do NOT start the seed loop yet.'
    )
print(f'PRE-FLIGHT OK — model responded: {resp[:80]!r}')
print(f'Model: {os.environ.get("OPENROUTER_MODEL")}')
print('Safe to start the seed loop below.')

---
## Cell 5: Run Seeds (one at a time)

Each seed takes ~10-40 minutes depending on API latency.
**Edit the `seeds_to_run` list** to control which seeds to execute.

Completed seed files are copied to Google Drive after each one finishes.

In [ ]:
import os, sys, asyncio, shutil, importlib, json

os.chdir('/content/recursive-autonomy-research')

# ====== EDIT THIS LIST ======
# Run whichever seeds you need. Already-completed seeds are skipped.
seeds_to_run = [42, 7, 13, 23, 88, 99, 101, 107, 113, 127]

# Google Drive backup path (set to None to skip backup)
DRIVE_BACKUP = '/content/drive/MyDrive/RAR_results'
if DRIVE_BACKUP:
    os.makedirs(DRIVE_BACKUP, exist_ok=True)

# Re-verify the API key hasn't been lost
key = os.environ.get('OPENROUTER_API_KEY', '')
if not key or key == 'YOUR_OPENROUTER_KEY_HERE':
    raise ValueError('API key missing! Re-run Cell 3.')

# Force reimport in case the module was already cached from a prior cell
if 'run_pilot_experiment' in sys.modules:
    del sys.modules['run_pilot_experiment']
if 'run_deep_learning_harness' in sys.modules:
    del sys.modules['run_deep_learning_harness']

import run_pilot_experiment as rpe

for seed in seeds_to_run:
    out_file = f'pilot_seed_{seed}.json'
    
    # Skip if already completed
    if os.path.exists(out_file):
        print(f'\n>>> Seed {seed}: ALREADY EXISTS ({out_file}), skipping.')
        continue
    
    print(f'\n'
          f'================================================================\n'
          f'  STARTING SEED {seed}\n'
          f'================================================================')
    
    os.environ['RAR_OUTPUT_FILE'] = out_file
    rpe.SEEDS = [seed]
    
    # Reset CYCLES in case it was changed
    rpe.CYCLES = int(os.environ.get('RAR_CYCLES', '60'))
    
    # Run the campaign for this single seed
    await rpe.execute_campaign()
    
    # Verify output was written
    if os.path.exists(out_file):
        size = os.path.getsize(out_file)
        print(f'\n>>> Seed {seed}: DONE -> {out_file} ({size:,} bytes)')
        
        # Backup to Google Drive
        if DRIVE_BACKUP:
            dst = os.path.join(DRIVE_BACKUP, out_file)
            shutil.copy2(out_file, dst)
            print(f'    Backed up to {dst}')
    else:
        print(f'\n>>> WARNING: Seed {seed} finished but {out_file} not found!')

print('\n\n========== ALL REQUESTED SEEDS COMPLETE ==========')

---
## Cell 6: Merge all seed files into final pilot_results.json

Run this **after all 10 seeds have completed** (across however many sessions it took).
If some seeds ran in earlier sessions, copy their `pilot_seed_*.json` files from
Google Drive back into this directory first.

In [ ]:
import os, glob, shutil

os.chdir('/content/recursive-autonomy-research')

# Restore any seed files from Google Drive that aren't in the local dir
DRIVE_BACKUP = '/content/drive/MyDrive/RAR_results'
if os.path.exists(DRIVE_BACKUP):
    for f in glob.glob(os.path.join(DRIVE_BACKUP, 'pilot_seed_*.json')):
        local = os.path.join('.', os.path.basename(f))
        if not os.path.exists(local):
            shutil.copy2(f, local)
            print(f'Restored {os.path.basename(f)} from Drive')

# List what we have
seed_files = sorted(glob.glob('pilot_seed_*.json'))
print(f'\nFound {len(seed_files)} seed files: {[os.path.basename(f) for f in seed_files]}')

if len(seed_files) < 2:
    print('Need at least 2 seeds to merge. Run more seeds first.')
else:
    !python merge_seeds.py
    
    # Backup merged result
    if os.path.exists('pilot_results.json') and os.path.exists(DRIVE_BACKUP):
        shutil.copy2('pilot_results.json', os.path.join(DRIVE_BACKUP, 'pilot_results.json'))
        print(f'\nMerged result backed up to {DRIVE_BACKUP}/pilot_results.json')

## Cell 7: Quick sanity check — show the honest numbers

In [ ]:
import json
with open('pilot_results.json') as f:
    r = json.load(f)

print(f"Seeds: {r['SEEDS']}")
print(f"Cycles: {r['CYCLES']}")
print(f"Wilcoxon p (RAR > Baseline): {r['wilcoxon_p_value_RAR_vs_Baseline']}")
print()

for cond in ['stateless_baseline', 'vector_rag', 'rar_compressed']:
    d = r['data']['conditions'][cond]
    ta = d['test_accuracies']
    nt = d['net_tokens']
    mean_ta = sum(ta)/len(ta)
    mean_nt = sum(nt)/len(nt)
    print(f'{cond:20s}  test_acc={mean_ta:.4f}  net_tokens={mean_nt:,.0f}')